# MeguvizModel · 中文短评识别第一跑

本笔记本在 **标准 CPU** 上训练一个字符片段分类器。它用来检查从加载样本到生成评估回执的流程，给后续 BGE、小模型微调和蒸馏提供对照。

1. 在 Colab 菜单 **代码执行程序 → 更改运行时类型** 中选择 **CPU**。如果出现 **高 RAM** 选项，关闭它。
2. 保留第 1 格的 **demo**，从上到下点代码左侧的 **▶**，也可以点顶部 **全部运行**。
3. 看到第 5 格的 **第一跑完成** 就可以收工。从左侧 **文件** 下载 JSON 回执，保存后断开运行时。

首次运行使用笔记本内置的虚构样例，无需装库、挂载 Google 云端硬盘或下载模型。样例人为区分了两种风格，只能用来检查流程。请用目标平台上有来源依据的独立数据评估实际误报。

## 1 · 选择数据

- **demo**：先跑这个。内置样例同时提供两种风格的成对文本。
- **detectrlx**：用于公开基准实验。会下载约 **690 MB**，核对文件哈希，从中文官方训练子集划分四个开发分区，保留官方测试子集。原始语料包含新闻、百科等领域。[DetectRL-X 数据卡](https://huggingface.co/datasets/WUJUNCHAO/DetectRL-X)
- **csv**：使用自有且有权使用的 UTF‑8 CSV。列名为 **group_id,text,label**，标签为 **human**、**ai** 或 **hybrid**。用同一 group_id 标记同一个原始种子、帖子或改写链里的样本。需要同时约束多个来源维度时，先把相互关联的样本归到同一组。混合写作样本会保留分组，退出二分类训练与校准。

In [ ]:
import csv
import hashlib
import json
import math
import shutil
import subprocess
import sys
import tempfile
import unicodedata
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path
from typing import NamedTuple

SOURCE = "demo"  # 首次保持 demo。可选 detectrlx 或 csv。
CSV_PATH = Path("/content/comments.csv")
MIN_VISIBLE = 11
MAX_CHARS = 128  # 这个基线按 Unicode 字符截断。BGE 的 token 长度另行验证。
SPLIT_SEED = "meguviz-model-01"
FPR_LIMIT = 0.005
ALPHA = 0.05

if "google.colab" in sys.modules and shutil.which("nvidia-smi"):
    probe = subprocess.run(
        ["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10
    )
    if probe.returncode == 0 and "UUID" in probe.stdout:
        raise RuntimeError("检测到 GPU。请把 Colab 切回标准 CPU 后重新运行。")

print(f"Python {sys.version.split()[0]} | CPU 分类器 | 数据模式 {SOURCE}")

## 2 · 加载并检查样本

代码会去掉空白分隔差异，记录太短的样本，并把仅有标点差异的文本视为潜在重复。公开基准用原始人工文本关联同源生成内容。

In [ ]:
class Sample(NamedTuple):
    group: str
    text: str
    label: str
    official_test: bool = False


def normalize(text):
    return " ".join(unicodedata.normalize("NFKC", text).split())


def fingerprint(text):
    content = "".join(
        char for char in normalize(text).casefold()
        if unicodedata.category(char)[0] not in "PZC"
    )
    return hashlib.sha256(content.encode("utf-8")).hexdigest() if content else ""


def visible_length(text):
    return sum(
        not char.isspace() and not unicodedata.category(char).startswith("C")
        for char in text
    )


def demo_samples():
    topics = """
    片头鼓点 夜景配乐 雨天收音 片尾字幕 横屏构图 街景调色 赛后采访 桌面走线
    咖啡磨豆 徒步路线 排队镜头 夜间猫叫 烘焙步骤 城市航拍 录屏画质 直播音量
    相机对焦 逆光镜头 旧城公交 料理特写 展览入口 有线耳机 推门转场 坡道骑行
    纸张纹理 键帽声音 雪地曝光 小狗表情 车窗倒影 弹幕速度 调弦过程 打印效果
    书店布景 路灯氛围 慢动作回放 黄昏天空 滑板落地 后台花絮 自行车刹车 露营帐篷
    公园开门 租房测量 柠檬切片 公交报站 晨练视频 灯带安装 砂锅开盖 工作台灯
    阳台种菜 收纳抽屉 送餐路线 琴谱位置 黑板笔迹 补光角度 颁奖表情 冲洗过程
    烛光聚会 玻璃水杯 窗边看书 施工地图 多机位剪辑 录音棚门口 灶台清理 座椅靠背
    """.split()
    informal = (
        "刚倒回去看了下{topic}，这一下我之前居然没注意",
        "{topic}这儿我拉回去又看了一眼，懂了懂了",
        "有没有人也在看{topic}啊，我在这停了好久",
        "我本来都划走了欸，{topic}又把我拉回来了",
    )
    formal = (
        "整体来看，{topic}与内容衔接自然，增强了观看体验。",
        "值得一提的是，{topic}丰富了表现层次，也体现出细节上的用心。",
        "从呈现方式来看，{topic}具有一定亮点，为整体增添了表现力。",
        "在细节处理方面，{topic}起到了较好的补充作用，值得关注。",
    )
    for index, topic in enumerate(topics):
        group = f"demo-{index:02}"
        yield Sample(group, informal[index % len(informal)].format(topic=topic), "human")
        yield Sample(group, formal[index % len(formal)].format(topic=topic), "ai")
    yield Sample("demo-short", "确实", "human")
    yield Sample("demo-short", "可以", "ai")


DATA_REVISION = "ccc9a0a74793dd3448e0da1650b430f5df1b0402"
DATA_SHA256 = "b1d6e839724dd7292421f7e9bd51d9d5dceb5bf87da31e8aab9534fc881b4a7f"
DATA_URL = (
    "https://huggingface.co/datasets/WUJUNCHAO/DetectRL-X/resolve/"
    f"{DATA_REVISION}/Binary/binary_general_64_open.json"
)


def detectrlx_samples():
    print("下载 DetectRL-X 公开文件（约 690 MB）…", flush=True)
    digest = hashlib.sha256()
    total = 0
    with tempfile.TemporaryFile(mode="w+b") as local:
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            while chunk := response.read(1024 * 1024):
                total += len(chunk)
                if total > 750_000_000:
                    raise ValueError("语料超过预期大小，请检查数据卡。")
                digest.update(chunk)
                local.write(chunk)
        if digest.hexdigest() != DATA_SHA256:
            raise ValueError("语料哈希不匹配，请检查下载结果。")
        local.seek(0)
        items = json.load(local)
    print(f"下载并校验完成，共 {total / 1_000_000:.1f} MB", flush=True)

    for item in items:
        if item["lang"] != "chinese":
            continue
        split = item["split"]
        if split not in {"train", "test"}:
            raise ValueError(f"未知的官方分区 {split!r}")
        human_text = item["human_written_text"]
        group = fingerprint(human_text)
        if group:
            yield Sample(group, human_text, "human", split == "test")
            yield Sample(group, item["llm_generated_text"], "ai", split == "test")


def csv_samples():
    if not CSV_PATH.is_file():
        raise FileNotFoundError(f"请把 CSV 加到 Colab 左侧文件栏，再检查路径 {CSV_PATH}")
    with CSV_PATH.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        if not {"group_id", "text", "label"}.issubset(reader.fieldnames or []):
            raise ValueError("CSV 要有 group_id、text、label 三列。")
        for line, row in enumerate(reader, start=2):
            group = (row["group_id"] or "").strip()
            text = row["text"] or ""
            label = (row["label"] or "").strip().lower()
            if not group or not text.strip() or label not in {"human", "ai", "hybrid"}:
                raise ValueError(f"CSV 第 {line} 行的分组、文本或标签有误。")
            yield Sample(group, text, label)


sources = {"demo": demo_samples, "detectrlx": detectrlx_samples, "csv": csv_samples}
if SOURCE not in sources:
    raise ValueError(f"数据模式请选择 {', '.join(sources)}。")

samples = []
short_count = 0
official_test_groups = set()
for sample in sources[SOURCE]():
    if sample.official_test:
        official_test_groups.add(sample.group)
    text = normalize(sample.text)[:MAX_CHARS]
    if visible_length(text) < MIN_VISIBLE or not fingerprint(text):
        short_count += 1
        continue
    samples.append(sample._replace(text=text))

if not samples:
    raise ValueError("未找到可用于分类的样本。请检查数据及短文本规则。")
print(f"候选样本 {len(samples)} 条 | 短文本或空内容 {short_count} 条")

## 3 · 按组隔离五个分区

| 分区 | 用途 |
| --- | --- |
| Train | 学习字符片段的权重 |
| Dev | 检查模型是否正常工作 |
| Cal-prob | 拟合分数校准温度 |
| Cal-threshold | 计算疑似 AI 的候选门槛 |
| Blind Test | 保留独立测试数据，本次只记录数量 |

相同文本会合并其原本所属的组。公开基准中，任何触及官方测试集的组都整组退出拟合，官方训练行不会计入测试集。

In [ ]:
parents = {row.group: row.group for row in samples}


def root(group):
    while group != parents[group]:
        parents[group] = parents[parents[group]]
        group = parents[group]
    return group


first_group_for_text = {}
labels_for_text = defaultdict(set)
for row in samples:
    key = fingerprint(row.text)
    previous = first_group_for_text.setdefault(key, row.group)
    left, right = sorted((root(previous), root(row.group)))
    parents[right] = left
    labels_for_text[key].add(row.label)

held_groups = {root(group) for group in official_test_groups if group in parents}
by_group = defaultdict(list)
seen = set()
same_text_cross_label = 0
train_rows_with_test_link = 0
for row in samples:
    key = fingerprint(row.text)
    group = root(row.group)
    if group in held_groups and not row.official_test:
        train_rows_with_test_link += 1
        continue
    if len(labels_for_text[key]) > 1:
        same_text_cross_label += 1
        continue
    if key not in seen:
        seen.add(key)
        by_group[group].append(row._replace(group=group))


def digest_string(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


names = ["Train", "Dev", "Cal-prob", "Cal-threshold", "Blind Test"]
splits = {name: [] for name in names}
training_groups = sorted(
    by_group.keys() - held_groups,
    key=lambda group: digest_string(f"{SPLIT_SEED}:{group}"),
)
n_groups = len(training_groups)
if SOURCE == "detectrlx":
    cuts = [n_groups * p // 100 for p in (60, 75, 85, 100)]
    splits["Blind Test"] = [
        row for group in sorted(held_groups) for row in by_group.get(group, [])
    ]
else:
    cuts = [n_groups * p // 100 for p in (60, 75, 85, 92, 100)]

for name, start, end in zip(names, [0, *cuts], cuts):
    splits[name] = [
        row for group in training_groups[start:end] for row in by_group[group]
    ]

partition_summary = {}
all_partition_groups = set()
for name, rows in splits.items():
    groups = {row.group for row in rows}
    assert all_partition_groups.isdisjoint(groups), "发现跨分区重复组"
    all_partition_groups.update(groups)
    summary = {
        "groups": len(groups),
        "rows": len(rows),
        "group_digest": digest_string(json.dumps(sorted(groups), ensure_ascii=False)),
    }
    if name != "Blind Test":
        summary["labels"] = dict(Counter(row.label for row in rows))
        if not {"human", "ai"}.issubset(summary["labels"]):
            raise ValueError(f"{name} 需要同时包含人工和 AI 标签，请增加独立标注组。")
    partition_summary[name] = summary
    print(f"{name:14} {len(rows):>6} 条，{len(groups):>6} 组")
print(f"已排除标签冲突 {same_text_cross_label} 条，测试组关联训练行 {train_rows_with_test_link} 条")

## 4 · 在 CPU 上训练基线

模型学习“回去看”“整体来”这类相邻字符片段在两类文本里出现的频率。这里用朴素贝叶斯快速跑通流程。Dev 的准确率只统计模型认识至少一个字符片段的样本，使用预先固定的 0.5 分数，仅作调试；上线候选门槛由下一格计算。

In [ ]:
def grams(text):
    return Counter(
        text[i:i + size]
        for size in (2, 3)
        for i in range(len(text) - size + 1)
    )


learned = {"human": Counter(), "ai": Counter()}
for row in splits["Train"]:
    if row.label in learned:
        learned[row.label].update(grams(row.text))

human = learned["human"]
ai = learned["ai"]
vocabulary = human.keys() | ai.keys()
human_total = sum(human.values()) + len(vocabulary)
ai_total = sum(ai.values()) + len(vocabulary)
weights = {
    gram: math.log((ai[gram] + 1) / ai_total) - math.log((human[gram] + 1) / human_total)
    for gram in vocabulary
}


def raw_score(text):
    contributions = [
        weights[gram] * count for gram, count in grams(text).items()
        if gram in weights
    ]
    return sum(contributions) if contributions else None


dev = [row for row in splits["Dev"] if row.label in learned]
dev_scores = [(raw_score(row.text), row.label) for row in dev]
scored_dev = [(score, label) for score, label in dev_scores if score is not None]
correct = sum((score >= 0.0) == (label == "ai") for score, label in scored_dev)
dev_accuracy = correct / len(scored_dev) if scored_dev else None
print(f"CPU 训练完成 | 学到 {len(weights)} 种字符片段 | Dev 可评分 {len(scored_dev)}/{len(dev)}")
if dev_accuracy is not None:
    print(f"Dev 固定门槛准确率 {dev_accuracy:.1%}")
if SOURCE == "demo":
    print("这是内置虚构样例的调试结果。")

## 5 · 校准分数并保存回执

先仅用 Cal-prob 拟合温度，再用 Cal-threshold 里的人工样本确定候选门槛。每个独立组取一个最高人工分数，以组为单位控制误报。单侧 95% 上界要压到 0.5%，即使观察到零个误报也至少需要 **598 个独立人工组**。数据不够时，有效长度的输入继续显示试验分数，状态为证据不足。

正式验收需要锁定分类器、预处理和门槛后，在独立目标平台盲测集上检查误报。

In [ ]:
prob_samples = []
for row in splits["Cal-prob"]:
    if row.label not in learned:
        continue
    score = raw_score(row.text)
    if score is not None:
        prob_samples.append((score, 1 if row.label == "ai" else -1))

if {sign for _, sign in prob_samples} != {-1, 1}:
    raise ValueError("Cal-prob 需要两类可评分样本。请补充与训练集同领域的独立标注组。")


def log_loss_at(temperature):
    return sum(
        max(value, 0) + math.log1p(math.exp(-abs(value)))
        for score, sign in prob_samples
        for value in (-sign * score / temperature,)
    ) / len(prob_samples)


temperature = min(
    (0.125, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0, 64.0),
    key=log_loss_at,
)
minimum_human_groups = math.ceil(math.log(ALPHA) / math.log1p(-FPR_LIMIT))


def allowed_false_flags(count):
    log_mass = count * math.log1p(-FPR_LIMIT)
    log_ratio = math.log(FPR_LIMIT) - math.log1p(-FPR_LIMIT)
    cdf = 0.0
    for errors in range(count):
        cdf += math.exp(log_mass)
        if cdf > ALPHA:
            return errors - 1
        log_mass += math.log(count - errors) - math.log(errors + 1) + log_ratio
    return count - 1


human_group_scores = {}
for row in splits["Cal-threshold"]:
    if row.label == "human":
        score = raw_score(row.text)
        if score is not None:
            human_group_scores[row.group] = max(human_group_scores.get(row.group, -math.inf), score)

n_human = len(human_group_scores)
allowed = allowed_false_flags(n_human)
threshold = None
if n_human >= minimum_human_groups and allowed >= 0:
    scores = sorted(human_group_scores.values(), reverse=True)
    threshold = math.nextafter(scores[allowed], math.inf)


def score_example(text):
    text = normalize(text)[:MAX_CHARS]
    if visible_length(text) < MIN_VISIBLE or not fingerprint(text):
        return "证据不足（文本过短）"
    raw = raw_score(text)
    if raw is None:
        return "证据不足（缺少训练中见过的字符片段）"
    probability = 1 / (1 + math.exp(-max(-700, min(700, raw / temperature))))
    if threshold is None:
        status = "证据不足（待校准门槛）"
    else:
        status = "疑似 AI（试验）" if raw >= threshold else "未达到疑似门槛"
    return f"{status} | 试验分数 {probability:.3f}"


report = {
    "source": SOURCE,
    "data_revision": DATA_REVISION if SOURCE == "detectrlx" else None,
    "split_seed": SPLIT_SEED,
    "partitions": partition_summary,
    "short_or_empty_rows": short_count,
    "cross_label_duplicate_rows": same_text_cross_label,
    "train_rows_linked_to_official_test": train_rows_with_test_link,
    "temperature": temperature,
    "dev_accuracy_at_0_5": dev_accuracy,
    "dev_scored_rows": len(scored_dev),
    "dev_binary_rows": len(dev),
    "eligible_human_groups_for_calibration": n_human,
    "minimum_human_groups_at_zero_errors": minimum_human_groups,
    "candidate_raw_threshold": threshold,
    "blind_evaluated": False,
}
report_path = Path.cwd() / "meguviz_colab_01_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"误报门槛校准 | 人工组 {n_human} | 零误报时至少需要 {minimum_human_groups} 组")
print("候选门槛状态 |", "样本不足，保留拒判" if threshold is None else "已计算，待独立盲测")
for example in ("确实", "拉回去看了两遍，那个小狗歪头真的好好笑啊"):
    print(f"{example} → {score_example(example)}")
print(f"第一跑完成 | 回执 {report_path.name} | 独立盲测未运行")